# 07 | Revisão da contagem RAND

No 06 usamos a contagem de registros RAND vigentes em 31 de dezembro. Aqui conferimos se as datas da planilha estão certas, o que de fato entra nessa contagem e se o resultado muda quando trocamos o dia de corte.

**Como usar:** execute antes 01 → 06. Depois rode os três blocos na ordem.

## Bloco 1: Datas de vigência

Cada registro tem uma data de início (`Effective Date`) e, quando foi substituído, uma data de fim (`Supersession Date`). Conferimos se tem data faltando ou invertida e em que época do ano as regras costumam começar a valer.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.association_data import count_rand_records
from src.association_model import fit_within_state

leis = pd.read_excel(ROOT / "reports/attachments/TL-A243-2-v4 State Firearm Law Database 6.0.xlsx", sheet_name="Database")
painel = pd.read_csv(ROOT / "data/processed/us_gun_policy_panel_2019_2024.csv")
estados = set(painel["state"])

def data(prefixo):
    partes = leis[[f"{prefixo} Year", f"{prefixo} Month", f"{prefixo} Day"]].copy()
    partes.columns = ["year", "month", "day"]
    return pd.to_datetime(partes, errors="coerce")

leis["inicio"] = data("Effective Date")
leis["fim"] = data("Supersession Date")
regras = leis[leis["State"].isin(estados) & leis["Type of Change"].isin(["Implement", "Modify"])].copy()
print("Registros Implement/Modify nos 50 estados:", len(regras))
print("Sem data de início:", regras["inicio"].isna().sum(), "| fim antes do início:", (regras["fim"] < regras["inicio"]).sum())

novos = regras[regras["inicio"].between("2019-01-01", "2024-12-31")]
dia = novos["inicio"].dt.strftime("%d/%m")
print("Começaram a valer entre 2019 e 2024:", len(novos))
print("  em 1º de janeiro:", (dia == "01/01").sum(), "| em 1º de julho:", (dia == "01/07").sum(),
      "| em outras datas:", ((dia != "01/01") & (dia != "01/07")).sum())
print("  no segundo semestre:", (novos["inicio"].dt.month >= 7).sum())
display(novos.groupby(novos["inicio"].dt.year)["Effect"].value_counts().unstack(fill_value=0))

Registros Implement/Modify nos 50 estados: 1781
Sem data de início: 0 | fim antes do início: 0
Começaram a valer entre 2019 e 2024: 157
  em 1º de janeiro: 21 | em 1º de julho: 34 | em outras datas: 102
  no segundo semestre: 82


Effect,Permissive,Restrictive
inicio,,
2019,4,29
2020,1,21
2021,8,12
2022,10,22
2023,4,20
2024,8,18


Não achamos data faltando nem invertida. Mas só 21 dos 157 registros novos começaram em 1º de janeiro, e mais da metade começou no segundo semestre. Ou seja, quase toda mudança acontece no meio de algum ano.

Com o corte em 31/12 do ano anterior, uma regra que começou em março só entra na conta do ano seguinte, mesmo tendo valido por dez meses. É uma escolha segura (a regra já existia antes do ano em que medimos os homicídios), mas ela atrasa o registro das mudanças. No bloco 3 testamos outros cortes.

## Bloco 2: O que entra na contagem

Olhamos quais tipos de regra formam a contagem e o que acontece quando uma lei é revogada.

In [2]:
vigentes = regras[(regras["inicio"] <= "2023-12-31") & (regras["fim"].isna() | (regras["fim"] > "2023-12-31"))]
print("Registros vigentes em 31/12/2023:", len(vigentes))
por_tipo = vigentes.groupby(["Law Class", "Effect"]).size().unstack(fill_value=0)
por_tipo["total"] = por_tipo.sum(axis=1)
display(por_tipo.sort_values("total", ascending=False))

revogacoes = leis[leis["State"].isin(estados) & (leis["Type of Change"] == "Repeal")
                  & leis["inicio"].between("2019-01-01", "2024-12-31")]
print("Revogações entre 2019 e 2024:", len(revogacoes))
display(revogacoes[["Law ID", "State", "Law Class", "Law Class Subtype", "inicio"]])
display(painel.loc[painel["state"] == "North Carolina", ["year", "restrictive_records", "permissive_records"]])

porte_ca = leis[(leis["State"] == "California") & (leis["Law Class"] == "carrying a concealed weapon (ccw)")]
display(porte_ca[["Law ID", "Law Class Subtype", "Effect", "Type of Change", "inicio", "fim"]])

Registros vigentes em 31/12/2023: 1202


Effect,Permissive,Restrictive,total
Law Class,,,
minimum age,11,293,304
prohibited possessor,0,185,185
background checks,1,163,164
firearms in college/university,10,61,71
local laws preempted by state,58,2,60
carrying a concealed weapon (ccw),51,1,52
castle doctrine,50,0,50
firearm sales restrictions,0,43,43
child access laws,0,36,36


Revogações entre 2019 e 2024: 6


,Law ID,State,Law Class,Law Class Subtype,inicio
560,IA1049,Iowa,waiting period,Waiting Periods: De facto (permit required),2021-07-01
1097,NC1057,North Carolina,background checks,sales from dealer,2023-03-29
1098,NC1058,North Carolina,background checks,permit to purchase,2023-03-29
1099,NC1059,North Carolina,background checks,private sales : point of sale,2023-03-29
1108,NC1069,North Carolina,permit to purchase,maximum waiting period,2023-03-29
1109,NC1070,North Carolina,waiting period,Waiting Periods: De facto (permit required),2023-03-29


,year,restrictive_records,permissive_records
192,2019,20,5
193,2020,20,5
194,2021,20,5
195,2022,20,5
196,2023,15,4
197,2024,15,4


,Law ID,Law Class Subtype,Effect,Type of Change,inicio,fim
112,CA1007,may issue,Permissive,Implement,1953-01-01,2022-06-23
178,CA1107,shall issue,Permissive,Modify,2022-06-23,NaT
188,CA1118,shall issue,Restrictive,Modify,2024-01-01,NaT


Três coisas que valem para a interpretação:

- **A contagem é puxada por poucos tipos.** Idade mínima, pessoas proibidas de ter arma e checagem de antecedentes somam quase dois terços dos registros restritivos. Os permissivos são quase todos de quatro tipos (preempção de leis locais, porte velado, legítima defesa em casa e porte aberto), que a maioria dos estados tem. Por isso a contagem de permissivos varia pouco.
- **Uma lei pode valer vários registros.** A Carolina do Norte acabou em março de 2023 com a licença para comprar pistola. Na RAND isso fechou cinco registros restritivos e um permissivo de uma vez. No nosso painel o estado cai de 20 para 15 por causa de uma única lei.
- **`Effect` mostra a direção da mudança, não o tipo da lei.** No porte velado da Califórnia, a regra antiga de licença a critério da autoridade (`may issue`) aparece como permissiva. A troca de 2022 também é permissiva e a de 2024 é restritiva, e as duas continuam vigentes. O mesmo assunto conta um registro de cada lado.

As revogações estão tratadas direito: o registro `Repeal` não entra na conta e o registro revogado recebe data de fim.

## Bloco 3: Trocar o dia de corte

Refazemos o modelo do 06 com a contagem em outras datas: em 1º de julho do mesmo ano dos homicídios e em 31/12 do mesmo ano. Também tiramos um estado de cada vez para ver se algum deles segura o resultado sozinho.

In [3]:
def resumo(nome, resultado):
    r = resultado.set_index("term")
    linha = {"medida": nome}
    for termo, chave in [("restrictive_previous_year", "restritivos"), ("permissive_previous_year", "permissivos")]:
        linha[chave] = r.loc[termo, "estimate"]
        linha[chave + "_min"] = r.loc[termo, "ci_low"]
        linha[chave + "_max"] = r.loc[termo, "ci_high"]
    return linha

colunas = ["restrictive_records", "permissive_records", "total_law_records"]
linhas = [resumo("31/12 do ano anterior (atual)", fit_within_state(painel))]
# fit_within_state sempre usa a linha do ano anterior. Para testar a contagem do
# mesmo ano, guardamos a contagem de t na linha de t-1. A última linha não é usada.
for nome, mes, dia_corte in [("1/7 do mesmo ano", 7, 1), ("31/12 do mesmo ano", 12, 31)]:
    contagem = count_rand_records(leis, estados, years=range(2020, 2026), month=mes, day=dia_corte)
    contagem["year"] -= 1
    outro = painel.drop(columns=colunas).merge(contagem, on=["state", "year"], validate="one_to_one")
    linhas.append(resumo(nome, fit_within_state(outro)))
sensibilidade = pd.DataFrame(linhas)
display(sensibilidade.round(3))

sem_um = pd.DataFrame([resumo(estado, fit_within_state(painel[painel["state"] != estado])) for estado in sorted(estados)])
fora_do_zero = ((sem_um["restritivos_min"] > 0) | (sem_um["restritivos_max"] < 0)
                | (sem_um["permissivos_min"] > 0) | (sem_um["permissivos_max"] < 0)).sum()
print("Tirando um estado por vez:")
print("  restritivos entre", round(sem_um["restritivos"].min(), 3), "e", round(sem_um["restritivos"].max(), 3))
print("  permissivos entre", round(sem_um["permissivos"].min(), 3), "e", round(sem_um["permissivos"].max(), 3))
print("  rodadas com intervalo fora do zero:", fora_do_zero, "de", len(sem_um))

usados = painel[painel["year"] <= 2023].sort_values(["state", "year"])
mudou = usados.groupby("state")[["restrictive_records", "permissive_records"]].diff().fillna(0).ne(0)
print("Mudanças que o modelo enxerga (contagens de 2019 a 2023):")
for coluna in ["restrictive_records", "permissive_records"]:
    print(" ", coluna, "->", mudou[coluna].sum(), "mudanças em", usados.loc[mudou[coluna], "state"].nunique(), "estados")

saida = ROOT / "reports/results/cutoff_sensitivity.csv"
sensibilidade.to_csv(saida, index=False)
print("Salvo:", saida.relative_to(ROOT))

,medida,restritivos,restritivos_min,restritivos_max,permissivos,permissivos_min,permissivos_max
0,31/12 do ano anterior (atual),0.071,-0.072,0.213,-0.251,-0.904,0.403
1,1/7 do mesmo ano,0.025,-0.157,0.208,0.360,-0.056,0.776
2,31/12 do mesmo ano,0.022,-0.152,0.196,0.040,-0.597,0.678


Tirando um estado por vez:
  restritivos entre 0.04 e 0.102
  permissivos entre -0.371 e 0.0
  rodadas com intervalo fora do zero: 0 de 50
Mudanças que o modelo enxerga (contagens de 2019 a 2023):
  restrictive_records -> 40 mudanças em 23 estados
  permissive_records -> 8 mudanças em 8 estados
Salvo: reports/results/cutoff_sensitivity.csv


O coeficiente dos registros restritivos fica perto de zero nos três cortes. O dos permissivos muda de sinal: negativo com o corte atual e positivo com o corte em 1º de julho. Em todos os casos o intervalo de 95% passa pelo zero, e tirar qualquer estado não muda isso.

O modelo também tem pouca mudança para trabalhar. A contagem de permissivos só mudou em 8 estados nos anos usados. Com tão pouca variação, o sinal do coeficiente depende de detalhes como o dia de corte. A conclusão do 06 continua: não dá para afirmar uma direção.